# House Price Prediction API — Flask Deployment

This notebook prepares, launches, and tests a Flask REST API around your **existing trained model**.

**Before running:** place your trained `.joblib` or `.pkl` model beside this notebook. If you trained with scaling/encoding, use the complete saved preprocessing pipeline. The API cannot infer your private dataset's column names or valid sample values; inspect the model and use real dataset values below.


## Cell 1 — Install dependencies
Run once in the active notebook environment.

In [ ]:
%pip install -r requirements.txt

## Cell 2 — Imports

In [ ]:
import os
import sys
import time
import json
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests


## Cell 3 — Project path and model selection
This uses the notebook's directory instead of assuming a fixed folder under the user's home directory.

In [ ]:
PROJECT_DIR = Path.cwd()
MODEL_SOURCE_NAME = None  # Set exact filename if more than one model file exists.

if MODEL_SOURCE_NAME:
    MODEL_SOURCE = PROJECT_DIR / MODEL_SOURCE_NAME
else:
    candidates = [
        path for path in PROJECT_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {".joblib", ".pkl"}
        and path.name != "model.joblib"
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            "Place exactly one trained .joblib/.pkl file beside this notebook, "
            "or set MODEL_SOURCE_NAME to its exact filename. "
            f"Found: {[path.name for path in candidates]}"
        )
    MODEL_SOURCE = candidates[0]

if not MODEL_SOURCE.is_file():
    raise FileNotFoundError(f"Model not found: {MODEL_SOURCE}")

print("Selected model:", MODEL_SOURCE.resolve())


## Cell 4 — Load and inspect the saved model

In [ ]:
model = joblib.load(MODEL_SOURCE)
feature_names = getattr(model, "feature_names_in_", None)
number_of_features = getattr(model, "n_features_in_", None)

print("Model type:", type(model).__name__)
print("Expected feature count:", number_of_features)
if feature_names is not None:
    print("Feature names in training order:")
    for index, name in enumerate(feature_names, start=1):
        print(f"{index}. {name}")
else:
    print("No feature names stored; list inputs must follow training order.")


## Cell 5 — Create deployment folder and copy the model

In [ ]:
DEPLOY_DIR = PROJECT_DIR / "House_Price_Flask"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = DEPLOY_DIR / "model.joblib"
joblib.dump(model, MODEL_PATH)
print("Deployment model:", MODEL_PATH)


## Cell 6 — Create app.py
The app code is written as a real Python module and includes the three API routes, feature validation, error handling, and model loading.

In [ ]:
APP_CODE = '"""House Price Prediction REST API.\n\nPlace your trained model or complete preprocessing pipeline at MODEL_PATH,\nor set the HOUSE_PRICE_MODEL environment variable to its path.\n"""\nfrom __future__ import annotations\n\nimport logging\nimport os\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom flask import Flask, jsonify, request\nfrom werkzeug.exceptions import BadRequest\n\nLOGGER = logging.getLogger(__name__)\nlogging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))\n\nDEFAULT_MODEL_PATH = Path(__file__).resolve().parent / "model.joblib"\nMODEL_PATH = Path(os.getenv("HOUSE_PRICE_MODEL", str(DEFAULT_MODEL_PATH))).expanduser()\n\n\ndef load_model(model_path: Path) -> Any:\n    """Load the previously trained estimator or preprocessing pipeline."""\n    if not model_path.is_file():\n        raise FileNotFoundError(\n            f"Model file not found: {model_path}. "\n            "Copy your trained .joblib/.pkl model here or set HOUSE_PRICE_MODEL."\n        )\n    return joblib.load(model_path)\n\n\ndef create_app(model: Any | None = None) -> Flask:\n    """Create the Flask application; model injection also enables unit testing."""\n    app = Flask(__name__)\n    app.config["JSON_SORT_KEYS"] = False\n\n    loaded_model = model if model is not None else load_model(MODEL_PATH)\n    feature_names = getattr(loaded_model, "feature_names_in_", None)\n    expected_count = getattr(loaded_model, "n_features_in_", None)\n\n    app.extensions["prediction_model"] = loaded_model\n    app.extensions["feature_names"] = (\n        [str(name) for name in feature_names] if feature_names is not None else None\n    )\n    app.extensions["expected_feature_count"] = (\n        int(expected_count) if expected_count is not None else None\n    )\n\n    @app.get("/")\n    def home():\n        return jsonify({\n            "status": "success",\n            "message": "House Price Prediction API is running.",\n            "endpoints": {\n                "health": "GET /health",\n                "prediction": "POST /predict"\n            }\n        }), 200\n\n    @app.get("/health")\n    def health():\n        return jsonify({\n            "status": "healthy",\n            "model_loaded": True\n        }), 200\n\n    def prepare_input(features: Any):\n        names = app.extensions["feature_names"]\n        count = app.extensions["expected_feature_count"]\n\n        if isinstance(features, dict):\n            if names is None:\n                raise ValueError(\n                    "This model has no stored feature names. "\n                    "Send \'features\' as an ordered numeric list."\n                )\n            missing = [name for name in names if name not in features]\n            extra = [name for name in features if name not in names]\n            if missing or extra:\n                raise ValueError(\n                    f"Feature mismatch. Missing: {missing}; unexpected: {extra}"\n                )\n            # Preserve training-time column order.\n            return pd.DataFrame([[features[name] for name in names]], columns=names)\n\n        if isinstance(features, list):\n            if count is not None and len(features) != count:\n                raise ValueError(\n                    f"Expected {count} features, received {len(features)}."\n                )\n            if not features:\n                raise ValueError("\'features\' list cannot be empty.")\n            try:\n                values = np.asarray(features, dtype=float)\n            except (TypeError, ValueError) as exc:\n                raise ValueError("List features must contain numeric values.") from exc\n            if values.ndim != 1 or not np.isfinite(values).all():\n                raise ValueError("Features must be a 1-D list of finite numbers.")\n            if names is not None:\n                return pd.DataFrame([values], columns=names)\n            return values.reshape(1, -1)\n\n        raise ValueError("\'features\' must be a JSON object or numeric list.")\n\n    @app.post("/predict")\n    def predict():\n        if not request.is_json:\n            return jsonify({"error": "Content-Type must be application/json."}), 415\n\n        try:\n            payload = request.get_json()\n        except BadRequest:\n            return jsonify({"error": "Request body contains invalid JSON."}), 400\n\n        if not isinstance(payload, dict) or "features" not in payload:\n            return jsonify({"error": "JSON body must contain a \'features\' field."}), 400\n\n        try:\n            model_input = prepare_input(payload["features"])\n            prediction = loaded_model.predict(model_input)\n            prediction_array = np.asarray(prediction).reshape(-1)\n\n            if prediction_array.size != 1:\n                raise ValueError("The model must return exactly one prediction.")\n\n            predicted_price = float(prediction_array[0])\n            if not np.isfinite(predicted_price):\n                raise ValueError("The model returned a non-finite prediction.")\n\n            return jsonify({\n                "status": "success",\n                "predicted_price": predicted_price\n            }), 200\n\n        except (TypeError, ValueError) as exc:\n            return jsonify({"error": str(exc)}), 400\n        except Exception:\n            LOGGER.exception("Prediction failed.")\n            return jsonify({"error": "Prediction failed due to an internal error."}), 500\n\n    return app\n\n\nif __name__ == "__main__":\n    application = create_app()\n    application.run(\n        host=os.getenv("FLASK_HOST", "127.0.0.1"),\n        port=int(os.getenv("FLASK_PORT", "5000")),\n        debug=False\n    )\n'
(DEPLOY_DIR / 'app.py').write_text(APP_CODE, encoding='utf-8')
print('Created app.py')

## Cell 7 — Create requirements.txt and README.md

In [ ]:
REQUIREMENTS = 'Flask>=3.0,<4.0\njoblib>=1.3\nnumpy>=1.26\npandas>=2.0\nscikit-learn>=1.3\npytest>=8.0\nrequests>=2.31\n'
README = '# House Price Prediction API\n\n## Purpose\nA Flask REST API that loads an existing trained house-price regression estimator\n(or a complete preprocessing + estimator pipeline), accepts JSON features, and\nreturns a predicted price.\n\n## Project layout\n```text\nHouse_Price_Flask_Revision/\n├── app.py\n├── model.joblib             # add your own trained model; not included\n├── requirements.txt\n├── README.md\n├── test_api.py\n└── test_api.ipynb\n```\n\n## 1. Set up the environment\n```bash\npython -m venv .venv\n# Windows:\n.venv\\Scripts\\activate\n# macOS/Linux:\nsource .venv/bin/activate\n\npython -m pip install -r requirements.txt\n```\n\n## 2. Add the trained model\nCopy your own trained `.joblib` or `.pkl` file into this folder and name it\n`model.joblib`. If preprocessing (encoding/scaling) was used during training,\nsave and deploy the complete fitted Pipeline, not just the final estimator.\n\nAlternatively set `HOUSE_PRICE_MODEL` to the absolute path of your model file.\n\n## 3. Run the API\n```bash\npython app.py\n```\nDefault base URL: `http://127.0.0.1:5000`\n\n## 4. Endpoints\n- `GET /` – API information\n- `GET /health` – health check\n- `POST /predict` – prediction endpoint\n\n### JSON input\nIf the fitted model has `feature_names_in_`, send a JSON object using the exact\nfeature names:\n```json\n{\n  "features": {\n    "actual_training_column_1": 1200,\n    "actual_training_column_2": 3\n  }\n}\n```\n\nIf feature names are not stored, send a numeric list in exact training order:\n```json\n{"features": [1200, 3]}\n```\nThe example names and values are placeholders. Replace them with actual columns\nand valid sample values from your training dataset.\n\n### Successful response\n```json\n{"status": "success", "predicted_price": 250000.0}\n```\nThe number above is illustrative, not a real model output.\n\n## 5. Test with curl\nHealth:\n```bash\ncurl http://127.0.0.1:5000/health\n```\nPrediction (replace keys and values with your actual model inputs):\n```bash\ncurl -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" -d "{\\"features\\":{\\"actual_training_column_1\\":1200,\\"actual_training_column_2\\":3}}"\n```\n\n## 6. Automated tests\nRun:\n```bash\npytest -q\n```\nThe included tests use a small dummy estimator to test routes and validation.\nThey do not replace testing with your actual trained model.\n\n## Notes\n- The app does not invent a model or training feature names.\n- Joblib files should only be loaded from trusted sources.\n- Keep `debug=False` outside local development.\n'
(DEPLOY_DIR / 'requirements.txt').write_text(REQUIREMENTS, encoding='utf-8')
(DEPLOY_DIR / 'README.md').write_text(README, encoding='utf-8')
print('Created requirements.txt and README.md')

## Cell 8 — Install project dependencies
Run this once for the project environment. Avoid reinstalling packages on every notebook execution.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(DEPLOY_DIR / "requirements.txt")],
    check=True
)
print("Dependencies installed.")


## Cell 9 — Start Flask and wait for readiness

In [ ]:
BASE_URL = "http://127.0.0.1:5000"
server_log_path = DEPLOY_DIR / "server.log"
server_log = open(server_log_path, "w", encoding="utf-8")

server_process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=str(DEPLOY_DIR),
    stdout=server_log,
    stderr=subprocess.STDOUT
)

ready = False
for _ in range(20):
    if server_process.poll() is not None:
        server_log.flush()
        raise RuntimeError(f"Flask stopped. Inspect {server_log_path}")
    try:
        health_response = requests.get(f"{BASE_URL}/health", timeout=2)
        if health_response.ok:
            ready = True
            break
    except requests.RequestException:
        time.sleep(0.5)

if not ready:
    raise RuntimeError(f"Flask did not become ready. Inspect {server_log_path}")

print("Server ready:", health_response.json())


## Cell 10 — Build a feature input template
The values below are placeholders only. Replace them with valid values from your original dataset before predicting.

In [ ]:
if feature_names is not None:
    sample_features = {str(name): 1 for name in feature_names}
    print(json.dumps({"features": sample_features}, indent=2))
else:
    if number_of_features is None:
        raise RuntimeError("Model does not expose n_features_in_; inspect training code.")
    sample_features = [1.0] * int(number_of_features)
    print(json.dumps({"features": sample_features}, indent=2))

print("Replace placeholder values with valid training-data values.")


## Cell 11 — Call the prediction endpoint
This cell sends one JSON request to the running Flask API.

In [ ]:
payload = {"features": sample_features}
prediction_response = requests.post(
    f"{BASE_URL}/predict",
    json=payload,
    timeout=10
)
print("HTTP status:", prediction_response.status_code)
print(json.dumps(prediction_response.json(), indent=2))
prediction_response.raise_for_status()


## Cell 12 — Test invalid input handling

In [ ]:
invalid_response = requests.post(
    f"{BASE_URL}/predict",
    json={},
    timeout=10
)
print("HTTP status:", invalid_response.status_code)
print("Response:", invalid_response.json())
assert invalid_response.status_code == 400
print("Validation test passed.")


## Cell 13 — Verify project files

In [ ]:
required_files = ["app.py", "model.joblib", "requirements.txt", "README.md"]
for filename in required_files:
    path = DEPLOY_DIR / filename
    print(f"{filename}: {'PASS' if path.is_file() else 'FAIL'}")
assert all((DEPLOY_DIR / name).is_file() for name in required_files)
print("Project file check passed.")


## Cell 14 — Stop the Flask process
Run this when API testing is complete.

In [ ]:
if server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait()

server_log.close()
print("Flask server stopped.")


## Notes and troubleshooting

- **Model file missing:** copy your actual trained model next to this notebook.
- **Feature mismatch:** use exact feature names and order from training.
- **Categorical data / scaling:** deploy the complete fitted preprocessing pipeline.
- **Connection refused:** confirm Flask started and port 5000 is available.
- **Placeholder inputs:** replace them with realistic values from the original dataset.

The notebook and tests can validate API structure, but the actual prediction can only be verified using your own trained model and valid feature values.
